# Phase 2B v3: Tree Speculation with Cross-Iteration KV Cache Persistence

**Changes vs v2:**
- `draft_populate_tree`: accepts persistent draft prefix cache + root logits. No prefix forward.
- `verify_tree`: accepts persistent target prefix cache. Only tree tokens as input.
- `tree_spec_decode`: owns both caches, extends by accepted tokens each iteration.

**Expected** (n=16,d=3): 197 ms/iter -> ~120-150 ms/iter, speedup 0.557x -> ~0.73-0.87x.

In [1]:
!pip install -q transformers accelerate

In [2]:
from huggingface_hub import notebook_login
notebook_login()

In [3]:
import torch
import torch.nn.functional as F
import numpy as np
import json
import time
import copy
from transformers import AutoModelForCausalLM, AutoTokenizer, DynamicCache

print(f"Torch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Torch: 2.10.0+cu128, CUDA: True
GPU: Tesla T4
Memory: 15.6 GB


In [4]:
from google.colab import files
uploaded = files.upload()

with open("swor_acceptance_vector.json") as f:
    swor_data = json.load(f)
ACC_VECTOR = swor_data["acceptance_vector"]
print(f"Acceptance vector p[1..{len(ACC_VECTOR)}] = {[round(x, 4) for x in ACC_VECTOR]}")

Saving swor_acceptance_vector.json to swor_acceptance_vector.json
Acceptance vector p[1..5] = [0.8459, 0.5745, 0.55, 0.3333, 0.1667]


In [5]:
TARGET_MODEL = "meta-llama/Llama-3.2-3B"
DRAFT_MODEL  = "meta-llama/Llama-3.2-1B"

tokenizer = AutoTokenizer.from_pretrained(TARGET_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

target_model = AutoModelForCausalLM.from_pretrained(
    TARGET_MODEL, torch_dtype=torch.float16, device_map="cuda"
)
target_model.eval()

draft_model = AutoModelForCausalLM.from_pretrained(
    DRAFT_MODEL, torch_dtype=torch.float16, device_map="cuda"
)
draft_model.eval()

print(f"Target: {TARGET_MODEL} ({sum(p.numel() for p in target_model.parameters())/1e9:.2f}B params)")
print(f"Draft : {DRAFT_MODEL} ({sum(p.numel() for p in draft_model.parameters())/1e9:.2f}B params)")

config.json:   0%|          | 0.00/844 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


...

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

Target: meta-llama/Llama-3.2-3B (3.21B params)
Draft : meta-llama/Llama-3.2-1B (1.24B params)


In [6]:
class TreeNode:
    __slots__ = ["children", "size"]
    def __init__(self, children=None):
        self.children = children or []
        self.size = 1 + sum(c.size for c in self.children)
    def flatten(self):
        out = [(-1, 0)]
        queue = [(self, 0)]
        while queue:
            node, my_idx = queue.pop(0)
            for child in node.children:
                child_idx = len(out)
                parent_depth = out[my_idx][1]
                out.append((my_idx, parent_depth + 1))
                queue.append((child, child_idx))
        return out
    def max_depth(self):
        if not self.children: return 0
        return 1 + max(c.max_depth() for c in self.children)

def sequoia_dp(acc_rates, max_size, max_depth, max_branch):
    N, D, B = max_size, max_depth, max_branch
    P = np.asarray(acc_rates, dtype=float)
    if P.ndim == 1:
        P = np.tile(P, (D - 1, 1))
    if P.shape[1] < B + 1:
        P = np.pad(P, ((0, 0), (0, B + 1 - P.shape[1])))
    P = P[:D - 1, :B + 1]
    T = np.full((N + 1, D + 1, B + 1), -np.inf)
    T_max = np.full((N + 1, D + 1), -np.inf)
    T[1, 1:, 0] = 1.0
    T_max[1, 1:] = 1.0
    best_new_node = {(1, d, 0): None for d in range(1, D + 1)}
    best_tree = {(1, d): TreeNode() for d in range(1, D + 1)}
    for n in range(2, N + 1):
        for d in range(2, D + 1):
            for b in range(1, B + 1):
                vals = np.full(n - 1, -np.inf)
                for m in range(1, n):
                    prev = T[n - m, d, b - 1]
                    subtree = T_max[m, d - 1]
                    if np.isfinite(prev) and np.isfinite(subtree):
                        vals[m - 1] = prev + P[D - d, b] * subtree
                if np.any(np.isfinite(vals)):
                    T[n, d, b] = np.max(vals)
                    best_new_node[n, d, b] = best_tree.get((np.argmax(vals) + 1, d - 1))
            T_max[n, d] = np.max(T[n, d, :])
            if np.isfinite(T_max[n, d]):
                best_b = int(np.argmax(T[n, d, :]))
                kids = []
                remaining = n
                for b_idx in range(best_b, 0, -1):
                    nc = best_new_node.get((remaining, d, b_idx))
                    if nc is None: break
                    kids.insert(0, nc)
                    remaining -= nc.size
                best_tree[n, d] = TreeNode(children=kids)
    return T_max, best_tree

P_FULL = ACC_VECTOR + [0.05, 0.05, 0.02]
T_max, best_trees = sequoia_dp(P_FULL, max_size=64, max_depth=6, max_branch=5)

c_t4 = 0.596
def t_n(n):
    if n <= 4: return 1.0
    if n <= 8: return 1.05
    if n <= 16: return 1.15
    if n <= 32: return 1.35
    return 1.70
def predicted_speedup(n, d):
    G = T_max[n, d]
    if not np.isfinite(G): return 0.0
    return G / (t_n(n) + d * c_t4)

print("DP-predicted speedups:")
for n, d in [(8, 3), (16, 3)]:
    print(f"  (n={n}, d={d}): G={T_max[n,d]:.3f}, S={predicted_speedup(n,d):.3f}x")

DP-predicted speedups:
  (n=8, d=3): G=3.722, S=1.312x
  (n=16, d=3): G=4.941, S=1.682x


In [7]:
def build_tree_layout(tree_root):
    layout = tree_root.flatten()
    parent = np.array([p for p, _ in layout], dtype=np.int64)
    depth  = np.array([d for _, d in layout], dtype=np.int64)
    return parent, depth

def tree_summary(tree_root):
    parent, depth = build_tree_layout(tree_root)
    children_of = {i: [] for i in range(len(parent))}
    for i, p in enumerate(parent):
        if p >= 0: children_of[int(p)].append(i)
    branching = [len(children_of[i]) for i in range(len(parent)) if children_of[i]]
    return {"size": len(parent), "depth": int(depth.max()),
            "avg_branch": float(np.mean(branching)) if branching else 0.0}

TREE_CONFIGS = [(8, 3), (16, 3)]
BENCHMARK_TREES = {}
for n, d in TREE_CONFIGS:
    tree = best_trees.get((n, d))
    if tree is None:
        print(f"WARNING: no tree for (n={n}, d={d})")
        continue
    BENCHMARK_TREES[(n, d)] = tree
    print(f"Tree (n={n}, d={d}): {tree_summary(tree)}, predicted={predicted_speedup(n,d):.3f}x")

PRIMARY = (16, 3)
SECONDARY = (8, 3)

Tree (n=8, d=3): {'size': 8, 'depth': 2, 'avg_branch': 2.3333333333333335}, predicted=1.312x
Tree (n=16, d=3): {'size': 16, 'depth': 2, 'avg_branch': 3.0}, predicted=1.682x


In [8]:
# ============================================================
# V3 RUNTIME: Cross-iteration persistent KV caches
# ============================================================
# Key changes from v2:
# 1. draft_populate_tree: NO prefix forward. Uses persistent cache + root logits.
# 2. verify_tree: tree-only forward with cached prefix KV.
# 3. tree_spec_decode: owns both caches, extends after each iteration.

def _fork_cache(cache):
    return copy.deepcopy(cache)

def _cache_seq_len(cache):
    return int(cache.get_seq_length())

_PROF = {
    "init_prefix_ms_sum": 0.0, "init_prefix_calls": 0,
    "draft_extend_ms_sum": 0.0, "draft_extend_calls": 0,
    "verify_ms_sum": 0.0, "verify_calls": 0,
    "cache_extend_ms_sum": 0.0, "cache_extend_calls": 0,
    "iter_count": 0,
}
def _prof_reset():
    for k in _PROF:
        _PROF[k] = 0 if isinstance(_PROF[k], int) else 0.0


@torch.inference_mode()
def draft_populate_tree(draft_prefix_cache, root_logits,
                        tree_parent, tree_depth, max_branch):
    """Populate tree using PERSISTENT draft prefix cache. No prefix forward.
    draft_prefix_cache is NOT modified — all branches fork from it."""
    n_nodes = len(tree_parent)
    device = root_logits.device
    tree_tokens = torch.zeros(n_nodes, dtype=torch.long, device=device)

    children_of = {i: [] for i in range(n_nodes)}
    for i, p in enumerate(tree_parent.tolist()):
        if int(p) >= 0:
            children_of[int(p)].append(i)

    def _is_interior(i):
        return len(children_of[i]) > 0

    interior_children_count = {
        i: sum(1 for c in children_of[i] if _is_interior(c))
        for i in range(n_nodes)
    }

    # Node 0 (virtual root) -> uses persistent prefix cache
    node_cache = {0: draft_prefix_cache}
    remaining = dict(interior_children_count)

    # Depth-1 children from root logits (already computed)
    root_kids = children_of[0]
    if root_kids:
        topk = torch.topk(root_logits, k=len(root_kids)).indices
        for c_idx, child in enumerate(root_kids):
            tree_tokens[child] = topk[c_idx]

    max_depth_val = int(tree_depth.max()) if n_nodes > 0 else 0
    for dep in range(1, max_depth_val):
        nodes_at_depth = [
            i for i in range(n_nodes)
            if int(tree_depth[i]) == dep and _is_interior(i)
        ]
        for node in nodes_at_depth:
            parent = int(tree_parent[node])
            pcache_src = node_cache[parent]
            remaining[parent] -= 1

            # ALWAYS fork from root to protect persistent cache
            if parent == 0 or remaining[parent] > 0:
                pcache = _fork_cache(pcache_src)
            else:
                pcache = pcache_src
                node_cache.pop(parent, None)

            this_tok = tree_tokens[node].view(1, 1)
            cache_pos = _cache_seq_len(pcache)
            cache_position = torch.arange(cache_pos, cache_pos + 1, device=device)

            torch.cuda.synchronize()
            t1 = time.perf_counter()
            d_out = draft_model(
                input_ids=this_tok,
                past_key_values=pcache,
                use_cache=True,
                cache_position=cache_position,
            )
            torch.cuda.synchronize()
            _PROF["draft_extend_ms_sum"] += (time.perf_counter() - t1) * 1000.0
            _PROF["draft_extend_calls"] += 1

            node_cache[node] = d_out.past_key_values
            node_logits = d_out.logits[0, -1, :]

            kids = children_of[node]
            topk = torch.topk(node_logits, k=len(kids)).indices
            for c_idx, child in enumerate(kids):
                tree_tokens[child] = topk[c_idx]

    # Clean up branch caches (NOT the prefix cache at node 0)
    for k in list(node_cache.keys()):
        if k != 0:
            del node_cache[k]
    return tree_tokens


@torch.inference_mode()
def verify_tree(target_prefix_cache, prefix_last_logits,
                tree_tokens, tree_parent, tree_depth):
    """Verify tree using PERSISTENT target prefix cache.
    Only tree tokens are passed as input (not the full prefix).
    prefix_last_logits: target prediction at last prefix position (for depth-1 nodes)."""
    n_nodes = len(tree_tokens)
    device = tree_tokens.device

    real_mask = tree_depth >= 1
    real_idx = torch.where(real_mask)[0].tolist()
    real_tokens = tree_tokens[real_idx]
    n_real = len(real_idx)

    if n_real == 0:
        return None

    old_to_new = {old: new for new, old in enumerate(real_idx)}

    # Fork cache so tree tokens don't pollute persistent cache
    verify_cache = _fork_cache(target_prefix_cache)
    cached_len = _cache_seq_len(verify_cache)

    # Attention mask: (1, 1, n_real, cached_len + n_real)
    # All tree tokens attend to ALL cached prefix positions.
    # Among tree tokens: ancestor-only rule.
    mask_2d = torch.zeros(n_real, cached_len + n_real, dtype=torch.bool, device=device)
    mask_2d[:, :cached_len] = True  # attend to all cached prefix

    for new_i, old_i in enumerate(real_idx):
        j = old_i
        while j != -1:
            if j in old_to_new:
                mask_2d[new_i, cached_len + old_to_new[j]] = True
            j = int(tree_parent[j])

    attn_mask_4d = torch.where(
        mask_2d.unsqueeze(0).unsqueeze(0),
        torch.tensor(0.0, dtype=torch.float16, device=device),
        torch.tensor(float("-inf"), dtype=torch.float16, device=device),
    )

    # Position IDs: depth d -> position cached_len - 1 + d
    real_depths = tree_depth[real_idx].tolist()
    tree_pos = torch.tensor(
        [cached_len - 1 + int(d) for d in real_depths], device=device
    )
    position_ids = tree_pos.unsqueeze(0)

    # Sequential cache positions for bookkeeping (distinct from position_ids for RoPE)
    cache_pos_seq = torch.arange(cached_len, cached_len + n_real, device=device)

    # Target forward: ONLY tree tokens, with cached prefix KV
    t_out = target_model(
        input_ids=real_tokens.unsqueeze(0),
        past_key_values=verify_cache,
        attention_mask=attn_mask_4d,
        position_ids=position_ids,
        cache_position=cache_pos_seq,
        use_cache=True,
    )

    # Extract predictions
    target_preds = torch.zeros(n_real, dtype=torch.long, device=device)
    for new_i, old_i in enumerate(real_idx):
        parent_old = int(tree_parent[old_i])
        if parent_old == 0:
            # Parent is virtual root: use prefix_last_logits
            target_preds[new_i] = prefix_last_logits.argmax()
        else:
            parent_new = old_to_new[parent_old]
            target_preds[new_i] = t_out.logits[0, parent_new, :].argmax()

    bonus_preds = {
        new_i: t_out.logits[0, new_i, :].argmax().item()
        for new_i in range(n_real)
    }

    del verify_cache
    return {
        "real_idx": real_idx,
        "old_to_new": old_to_new,
        "target_preds": target_preds,
        "bonus_preds": bonus_preds,
    }


def accept_longest_path(tree_tokens, tree_parent, tree_depth, verify_result):
    """Walk from virtual root, accept longest matching path."""
    if verify_result is None:
        return [], None
    real_idx = verify_result["real_idx"]
    old_to_new = verify_result["old_to_new"]
    target_preds = verify_result["target_preds"].tolist()
    bonus_preds = verify_result["bonus_preds"]

    children_of = {}
    for i in range(len(tree_parent)):
        children_of[i] = []
    for i, p in enumerate(tree_parent.tolist()):
        if int(p) >= 0:
            children_of[int(p)].append(i)

    accepted = []
    current = 0
    last_real_new = None
    while True:
        kids = children_of.get(current, [])
        accepted_kid = None
        for kid in kids:
            if kid not in old_to_new:
                continue
            new_i = old_to_new[kid]
            if int(tree_tokens[kid]) == target_preds[new_i]:
                accepted_kid = kid
                accepted.append(int(tree_tokens[kid]))
                last_real_new = new_i
                break
        if accepted_kid is None:
            break
        current = accepted_kid

    bonus = bonus_preds[last_real_new] if last_real_new is not None else None
    return accepted, bonus


@torch.inference_mode()
def extend_caches(new_tokens, draft_cache, target_cache):
    """Extend both prefix caches with accepted + bonus tokens.
    Returns (draft_cache, target_cache, target_last_logits, draft_last_logits)."""
    device = "cuda"
    tokens = torch.tensor([new_tokens], device=device)
    n = len(new_tokens)

    # Draft cache extension
    d_len = _cache_seq_len(draft_cache)
    d_pos = torch.arange(d_len, d_len + n, device=device)
    d_out = draft_model(
        input_ids=tokens, past_key_values=draft_cache,
        use_cache=True, cache_position=d_pos,
    )
    draft_cache = d_out.past_key_values
    draft_logits = d_out.logits[0, -1, :]

    # Target cache extension
    t_len = _cache_seq_len(target_cache)
    t_pos = torch.arange(t_len, t_len + n, device=device)
    t_out = target_model(
        input_ids=tokens, past_key_values=target_cache,
        use_cache=True, cache_position=t_pos,
    )
    target_cache = t_out.past_key_values
    target_logits = t_out.logits[0, -1, :]

    return draft_cache, target_cache, target_logits, draft_logits


@torch.inference_mode()
def tree_spec_decode(prompt, tree_root, max_new_tokens=128):
    """Tree speculative decoding with persistent KV caches (v3)."""
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to("cuda")
    prefix_len_orig = input_ids.shape[1]

    parent_arr, depth_arr = build_tree_layout(tree_root)
    tree_parent = torch.tensor(parent_arr, device="cuda")
    tree_depth = torch.tensor(depth_arr, device="cuda")

    # Build initial prefix caches (one-time cost)
    torch.cuda.synchronize()
    t_init = time.perf_counter()

    draft_cache = DynamicCache()
    d_init = draft_model(input_ids=input_ids, past_key_values=draft_cache, use_cache=True)
    draft_cache = d_init.past_key_values
    draft_root_logits = d_init.logits[0, -1, :]

    target_cache = DynamicCache()
    t_init_out = target_model(input_ids=input_ids, past_key_values=target_cache, use_cache=True)
    target_cache = t_init_out.past_key_values
    prefix_last_logits = t_init_out.logits[0, -1, :]

    torch.cuda.synchronize()
    _PROF["init_prefix_ms_sum"] += (time.perf_counter() - t_init) * 1000.0
    _PROF["init_prefix_calls"] += 1

    n_iter = 0
    total_accepted = 0
    generated = []

    while len(generated) < max_new_tokens:
        # Draft: populate tree using persistent cache
        tree_tokens = draft_populate_tree(
            draft_cache, draft_root_logits,
            tree_parent, tree_depth, max_branch=5
        )

        # Verify: using persistent target cache
        torch.cuda.synchronize()
        t_v = time.perf_counter()
        verify_res = verify_tree(
            target_cache, prefix_last_logits,
            tree_tokens, tree_parent, tree_depth
        )
        torch.cuda.synchronize()
        _PROF["verify_ms_sum"] += (time.perf_counter() - t_v) * 1000.0
        _PROF["verify_calls"] += 1

        # Accept
        accepted, bonus = accept_longest_path(tree_tokens, tree_parent, tree_depth, verify_res)

        new_tokens = accepted.copy()
        if bonus is not None:
            new_tokens.append(bonus)
        else:
            new_tokens.append(int(prefix_last_logits.argmax()))

        if not new_tokens:
            break
        generated.extend(new_tokens)

        # Extend caches with accepted tokens
        torch.cuda.synchronize()
        t_ext = time.perf_counter()
        draft_cache, target_cache, prefix_last_logits, draft_root_logits =             extend_caches(new_tokens, draft_cache, target_cache)
        torch.cuda.synchronize()
        _PROF["cache_extend_ms_sum"] += (time.perf_counter() - t_ext) * 1000.0
        _PROF["cache_extend_calls"] += 1

        n_iter += 1
        _PROF["iter_count"] += 1
        total_accepted += len(accepted)

        if tokenizer.eos_token_id in new_tokens:
            break

    gen_ids = generated[:max_new_tokens]
    if gen_ids:
        output_ids = torch.cat([input_ids, torch.tensor([gen_ids], device="cuda")], dim=1)
    else:
        output_ids = input_ids
    return output_ids, n_iter, total_accepted

print("V3 runtime defined.")
print("Changes: no prefix forward, tree-only verify, persistent caches.")

V3 runtime defined.
Changes: no prefix forward, tree-only verify, persistent caches.


In [9]:
print("=" * 60)
print("CORRECTNESS CHECK 1: greedy baseline")
print("=" * 60)
test_prompt = "The history of artificial intelligence began in the 1950s when researchers first"
test_input = tokenizer(test_prompt, return_tensors="pt").to("cuda")
with torch.inference_mode():
    baseline_out = target_model.generate(
        **test_input, max_new_tokens=32,
        do_sample=False, pad_token_id=tokenizer.pad_token_id,
    )
baseline_ids = baseline_out[0].tolist()
prefix_len = test_input['input_ids'].shape[1]
print(f"Baseline first 20: {baseline_ids[prefix_len:prefix_len+20]}")
print(f"Text: {tokenizer.decode(baseline_ids[prefix_len:])[:150]}...")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


CORRECTNESS CHECK 1: greedy baseline
Baseline first 20: [6137, 311, 13488, 279, 4623, 315, 6968, 12933, 430, 1436, 1781, 323, 4048, 1093, 12966, 13, 8876, 1243, 11, 15592]
Text:  began to explore the idea of creating machines that could think and learn like humans. Since then, AI has come a long way, and today, it is used...


In [10]:
print("=" * 60)
print("CORRECTNESS CHECK 2: V3 tree vs baseline (20/20 required)")
print("=" * 60)

for cfg_name, cfg in [("PRIMARY", PRIMARY), ("SECONDARY", SECONDARY)]:
    _prof_reset()
    tree = BENCHMARK_TREES[cfg]
    tree_ids, n_iter, n_acc = tree_spec_decode(test_prompt, tree, max_new_tokens=32)
    tree_gen = tree_ids[0, prefix_len:prefix_len+20].tolist()
    baseline_gen = baseline_ids[prefix_len:prefix_len+20]

    n_match = sum(1 for a, b in zip(baseline_gen, tree_gen) if a == b)
    print(f"\n{cfg_name} {cfg}:")
    print(f"  Baseline: {baseline_gen}")
    print(f"  Tree:     {tree_gen}")
    print(f"  Match: {n_match}/20, iters={n_iter}, accepted={n_acc}")

    if _PROF["iter_count"] > 0:
        d_ext = _PROF["draft_extend_ms_sum"] / _PROF["iter_count"]
        v_ms = _PROF["verify_ms_sum"] / _PROF["iter_count"]
        c_ext = _PROF["cache_extend_ms_sum"] / _PROF["iter_count"]
        print(f"  Per-iter: draft_ext={d_ext:.1f}ms, verify={v_ms:.1f}ms, cache_ext={c_ext:.1f}ms, total={d_ext+v_ms+c_ext:.1f}ms")

    if n_match < 18:
        print(f"  *** FAIL *** Debug before proceeding!")
    else:
        print(f"  PASS")

CORRECTNESS CHECK 2: V3 tree vs baseline (20/20 required)

PRIMARY (16, 3):
  Baseline: [6137, 311, 13488, 279, 4623, 315, 6968, 12933, 430, 1436, 1781, 323, 4048, 1093, 12966, 13, 8876, 1243, 11, 15592]
  Tree:     [6137, 311, 13488, 279, 4623, 315, 6968, 12933, 430, 1436, 1781, 323, 4048, 1093, 12966, 13, 8876, 1243, 11, 15592]
  Match: 20/20, iters=12, accepted=22
  Per-iter: draft_ext=104.3ms, verify=53.6ms, cache_ext=67.4ms, total=225.3ms
  PASS

SECONDARY (8, 3):
  Baseline: [6137, 311, 13488, 279, 4623, 315, 6968, 12933, 430, 1436, 1781, 323, 4048, 1093, 12966, 13, 8876, 1243, 11, 15592]
  Tree:     [6137, 311, 13488, 279, 4623, 315, 6968, 12933, 430, 1436, 1781, 323, 4048, 1093, 12966, 13, 8876, 1243, 11, 15592]
  Match: 20/20, iters=12, accepted=22
  Per-iter: draft_ext=49.2ms, verify=46.5ms, cache_ext=63.1ms, total=158.7ms
  PASS


In [11]:
PROMPTS = [
    "The history of artificial intelligence began in the 1950s when researchers first",
    "In computer science, a hash table is a data structure that implements an associative",
    "The process of photosynthesis converts light energy into chemical energy through",
    'def fibonacci(n):\n    """Calculate the nth Fibonacci number."""\n    if n <= 1:\n        return n\n',
    "# Python implementation of binary search\ndef binary_search(arr, target):\n",
    "Summarize the following: Machine learning is a subset of artificial intelligence that focuses on building systems that learn from data. Machine learning algorithms use historical data as input to predict new output values. Machine learning is",
    "Write a detailed explanation of how TCP/IP networking works, starting from",
    "Explain the difference between a stack and a queue data structure. A stack",
    "The transformer architecture was introduced in the paper Attention is All You Need by Vaswani et al. in 2017. It replaced recurrent neural networks with self-attention mechanisms, enabling parallel processing of sequences. The key innovation was the multi-head attention mechanism, which allows the model to attend to different parts of the input simultaneously. Since then, transformers have become the foundation for",
    "Large language models are trained on massive datasets of text from the internet. During training, the model learns to predict the next token in a sequence given all previous tokens. This autoregressive training objective means that at inference time, the model generates text one token at a time, each conditioned on all previously generated tokens. This sequential nature creates a fundamental bottleneck because",
]
MAX_NEW_TOKENS = 128
NUM_WARMUP = 3
NUM_TRIALS = 30
BASELINE_TPS = 26.42

def run_tree_benchmark(tree_root, name):
    results = []
    for _ in range(NUM_WARMUP):
        tree_spec_decode(PROMPTS[0], tree_root, max_new_tokens=16)
    torch.cuda.reset_peak_memory_stats()
    _prof_reset()
    for trial in range(NUM_TRIALS):
        pidx = trial % len(PROMPTS)
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        out_ids, n_iter, n_acc = tree_spec_decode(PROMPTS[pidx], tree_root, MAX_NEW_TOKENS)
        torch.cuda.synchronize()
        wall = time.perf_counter() - t0
        prompt_len = tokenizer(PROMPTS[pidx], return_tensors="pt").input_ids.shape[1]
        gen_tokens = out_ids.shape[1] - prompt_len
        results.append({
            "method": name, "prompt_idx": pidx, "gen_tokens": gen_tokens,
            "wall_sec": round(wall, 5),
            "tok_per_sec": round(gen_tokens / wall, 2) if wall > 0 else 0,
            "n_iter": n_iter, "n_accepted": n_acc,
            "tokens_per_iter": round(gen_tokens / n_iter, 2) if n_iter > 0 else 0,
            "peak_gpu_mb": round(torch.cuda.max_memory_allocated() / 1e6, 1),
        })
        if trial % 10 == 0:
            print(f"  trial {trial}: {results[-1]['tok_per_sec']:.2f} tok/s")
    prof = {
        "draft_extend_ms_per_iter": _PROF["draft_extend_ms_sum"] / max(_PROF["iter_count"], 1),
        "draft_extend_ms_per_call": _PROF["draft_extend_ms_sum"] / max(_PROF["draft_extend_calls"], 1),
        "draft_extend_calls_per_iter": _PROF["draft_extend_calls"] / max(_PROF["iter_count"], 1),
        "verify_ms_per_iter": _PROF["verify_ms_sum"] / max(_PROF["verify_calls"], 1),
        "cache_extend_ms_per_iter": _PROF["cache_extend_ms_sum"] / max(_PROF["cache_extend_calls"], 1),
        "iter_count": _PROF["iter_count"],
    }
    return results, prof

def summarize(results):
    tps = [r["tok_per_sec"] for r in results]
    tpi = [r["tokens_per_iter"] for r in results]
    walls = [r["wall_sec"] for r in results]
    return {
        "method": results[0]["method"],
        "mean_tok_sec": round(float(np.mean(tps)), 2),
        "median_tok_sec": round(float(np.median(tps)), 2),
        "std_tok_sec": round(float(np.std(tps)), 2),
        "mean_tokens_per_iter": round(float(np.mean(tpi)), 2),
        "p95_wall_sec": round(float(np.percentile(walls, 95)), 4),
        "speedup_vs_baseline": round(float(np.mean(tps)) / BASELINE_TPS, 3),
        "peak_gpu_mb": max(r["peak_gpu_mb"] for r in results),
        "n_trials": len(results),
    }
print("Benchmark harness ready.")

Benchmark harness ready.


In [12]:
ALL_RESULTS = []
ALL_SUMMARIES = []
PROF_SNAPSHOTS = {}

for label, cfg in [("SECONDARY", SECONDARY), ("PRIMARY", PRIMARY)]:
    n, d = cfg
    print("=" * 60)
    print(f"BENCHMARK: tree (n={n}, d={d}) [{label}]")
    print(f"Predicted speedup: {predicted_speedup(n, d):.3f}x")
    print("=" * 60)
    tree = BENCHMARK_TREES[cfg]
    res, prof = run_tree_benchmark(tree, f"tree_n{n}_d{d}")
    ALL_RESULTS.extend(res)
    s = summarize(res)
    s["predicted_speedup"] = round(predicted_speedup(n, d), 3)
    s["tree_config"] = f"n={n}, d={d}"
    ALL_SUMMARIES.append(s)
    PROF_SNAPSHOTS[f"n{n}_d{d}"] = prof
    print(f"\nResult: {s['mean_tok_sec']:.2f} tok/s | speedup={s['speedup_vs_baseline']:.3f}x")
    d_ms = prof['draft_extend_ms_per_iter']
    v_ms = prof['verify_ms_per_iter']
    c_ms = prof['cache_extend_ms_per_iter']
    print(f"  draft_ext={d_ms:.1f} + verify={v_ms:.1f} + cache_ext={c_ms:.1f} = {d_ms+v_ms+c_ms:.1f} ms/iter")
    with open("phase2b_v3_partial.json", "w") as f:
        json.dump({"summaries": ALL_SUMMARIES, "prof": PROF_SNAPSHOTS}, f, indent=2)
    print()

BENCHMARK: tree (n=8, d=3) [SECONDARY]
Predicted speedup: 1.312x
  trial 0: 15.34 tok/s
  trial 10: 14.86 tok/s
  trial 20: 16.70 tok/s

Result: 14.90 tok/s | speedup=0.564x
  draft_ext=59.8 + verify=55.7 + cache_ext=75.8 = 191.3 ms/iter

BENCHMARK: tree (n=16, d=3) [PRIMARY]
Predicted speedup: 1.682x
  trial 0: 12.37 tok/s
  trial 10: 12.78 tok/s
  trial 20: 11.75 tok/s

Result: 12.05 tok/s | speedup=0.456x
  draft_ext=106.0 + verify=51.2 + cache_ext=67.2 = 224.5 ms/iter



In [13]:
print("=" * 70)
print("V3 RESULTS")
print("=" * 70)
print(f"{'Config':<16} {'Tok/s':>8} {'Speedup':>9} {'Predicted':>10} {'Tok/iter':>9}")
for s in ALL_SUMMARIES:
    print(f"{s['tree_config']:<16} {s['mean_tok_sec']:>8.2f} {s['speedup_vs_baseline']:>8.3f}x {s['predicted_speedup']:>9.3f}x {s['mean_tokens_per_iter']:>9.2f}")

print("\n--- Per-iter timing ---")
for key, p in PROF_SNAPSHOTS.items():
    d = p["draft_extend_ms_per_iter"]
    v = p["verify_ms_per_iter"]
    c = p["cache_extend_ms_per_iter"]
    print(f"{key}: draft_ext={d:.1f} + verify={v:.1f} + cache_ext={c:.1f} = {d+v+c:.1f} ms")

print("\n--- V2 -> V3 ---")
print("V2 n16_d3: prefix=32.9 + extend=95.2 + verify=68.9 = 197.0 ms | 0.557x")
if "n16_d3" in PROF_SNAPSHOTS:
    p = PROF_SNAPSHOTS["n16_d3"]
    t = p["draft_extend_ms_per_iter"] + p["verify_ms_per_iter"] + p["cache_extend_ms_per_iter"]
    s16 = [s for s in ALL_SUMMARIES if "n=16" in s["tree_config"]]
    sx = s16[0]["speedup_vs_baseline"] if s16 else 0
    print(f"V3 n16_d3: ext={p['draft_extend_ms_per_iter']:.1f} + verify={p['verify_ms_per_iter']:.1f} + cache_ext={p['cache_extend_ms_per_iter']:.1f} = {t:.1f} ms | {sx:.3f}x")
    print(f"Saved: {197.0-t:.1f} ms/iter ({(197.0-t)/197.0*100:.0f}%)")

print("\nAll methods:")
print("Baseline AR   : 26.42 tok/s | 1.000x")
print("Linear SD k=3 : 26.69 tok/s | 1.010x")
print("PLD n=10      : 36.67 tok/s | 1.388x")
for s in ALL_SUMMARIES:
    print(f"V3 Tree {s['tree_config']:<9}: {s['mean_tok_sec']:5.2f} tok/s | {s['speedup_vs_baseline']:.3f}x")

V3 RESULTS
Config              Tok/s   Speedup  Predicted  Tok/iter
n=8, d=3            14.90    0.564x     1.312x      2.86
n=16, d=3           12.05    0.456x     1.682x      2.94

--- Per-iter timing ---
n8_d3: draft_ext=59.8 + verify=55.7 + cache_ext=75.8 = 191.3 ms
n16_d3: draft_ext=106.0 + verify=51.2 + cache_ext=67.2 = 224.5 ms

--- V2 -> V3 ---
V2 n16_d3: prefix=32.9 + extend=95.2 + verify=68.9 = 197.0 ms | 0.557x
V3 n16_d3: ext=106.0 + verify=51.2 + cache_ext=67.2 = 224.5 ms | 0.456x
Saved: -27.5 ms/iter (-14%)

All methods:
Baseline AR   : 26.42 tok/s | 1.000x
Linear SD k=3 : 26.69 tok/s | 1.010x
PLD n=10      : 36.67 tok/s | 1.388x
V3 Tree n=8, d=3 : 14.90 tok/s | 0.564x
V3 Tree n=16, d=3: 12.05 tok/s | 0.456x


In [14]:
output = {
    "version": "v3_persistent_cache",
    "target_model": TARGET_MODEL, "draft_model": DRAFT_MODEL,
    "max_new_tokens": MAX_NEW_TOKENS, "n_trials": NUM_TRIALS,
    "baseline_tps": BASELINE_TPS, "cost_ratio_c": c_t4,
    "acceptance_vector": ACC_VECTOR,
    "configs": [PRIMARY, SECONDARY],
    "raw_results": ALL_RESULTS, "summaries": ALL_SUMMARIES,
    "profiler_snapshots": PROF_SNAPSHOTS,
    "v2_comparison": {"v2_n16_d3": 0.557, "v2_n8_d3": 0.771, "v2_iter_ms": 197.0},
}
with open("phase2b_v3_results.json", "w") as f:
    json.dump(output, f, indent=2)
print("Saved phase2b_v3_results.json")
try:
    from google.colab import files
    files.download("phase2b_v3_results.json")
except:
    pass

Saved phase2b_v3_results.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>